<!-- [intro-a] -->

# ParallelProse — v2, Final Architecture

`v2-01` named six new pieces at arm’s length, before any of them existed; `v2-02` sketched their shapes in code that wasn’t wired in yet. All five of v2’s roadmap items — the merge, the nested per-corpus state, `reflect` as a diagnostic router, per-corpus retry, and composer’s structured output — are now built and validated. This notebook walks the same six letters again, but this time pointing at the real, finished `agent.py`.

Two fields visible in the code below — `era` and `quote` — are v3 additions layered on top of this same skeleton after v2 closed; they’re not part of v2’s own architecture and aren’t covered here.

<!-- [intro-b] -->

Where each piece landed, final:

| Section | Piece | Roadmap item | Status |
|---|---|---|---|
| B | one slot per book in the shared state | 2 | done, 2026-09-24 |
| C | a tool-selecting retriever, per book | 1, 2 | done, 2026-09-19 / 2026-09-24 |
| D | `reflect` judging each book separately | 4 | done, 2026-09-27 |
| E | retry aimed at only the flagged book | 5 | done, 2026-09-28 |
| F | a structured comparison from `composer` | 6 | done, 2026-10-02 |

<!-- [A-a] -->

## A — The big picture, built

v1 ran two separate agents: a fixed reflect-loop graph for one book, and a tool-flexible demo agent with no quality loop. v2 merged them into the one graph below, now compiled and running.

<!-- [A.1a] -->

```mermaid
flowchart LR
    START --> RET["retriever"]
    RET --> COMP["composer"]
    COMP --> REF["reflect"]
    REF -->|"a book is miss"| RET
    REF -->|"no book is miss"| END
```

<!-- [A.2a] -->

That’s the real shape `reflection_graph` compiles into (agent.py:334-343): three nodes, one conditional edge out of `reflect`. `retriever`, `composer`, and `reflect` are the same three nodes v1 had — nothing moved. What changed is what each one does now that there are two books instead of one, which is what sections B through F below are each about.

<!-- [B-a] -->

## B — The state that carries two books, built

Before either book gets touched, the state needs somewhere to keep two books’ worth of notes without mixing them up.

In [1]:
# [B.1a]
from ParallelProse.agent import CorpusState, ReflectionState

print('CorpusState fields:', list(CorpusState.__annotations__))
print('ReflectionState fields:', list(ReflectionState.__annotations__))

CorpusState fields: ['chunks', 'lineage', 'label', 'reason', 'narrowed_query']
ReflectionState fields: ['query', 'answer', 'needs_revision', 'feedback', 'iteration', 'corpora']


<!-- [B.2a] -->

`corpora` (agent.py:40-46) is the one new field on `ReflectionState` — a dict keyed `"A"`/`"B"`, each value a full `CorpusState` (agent.py:32-37). Every other field (`query`, `answer`, `feedback`, `iteration`) stays exactly as flat as it was in v1; only the per-book concerns moved under `corpora`. Which two books fill those two keys is itself data, not code — `catalog.py`’s `CATALOG` dict holds one `Book` entry per corpus (title, author, collection, file), so swapping a book is a one-entry edit, not a code change.

<!-- [C-a] -->

## C — retriever, a tool-selecting agent per book, built

v1’s `retriever` hardcoded one retrieval call. v2’s needs to pick among several retrieval strategies, per book, and skip a book entirely once it’s already been judged good.

<!-- [C.1a] -->

`retriever` (agent.py:96-179) loops over `state["corpora"]`. For each book, it builds (or reuses, from `bridged_tools_by_corpus`/`retrieval_agents_by_corpus`, agent.py:28-29) a small inner `create_agent` bounded by `ToolCallLimitMiddleware` and `ModelCallLimitMiddleware` — that’s the tool-flexible half of v1 the merge kept. Then it checks that book’s own `label`: `None` (round 1) or `"miss"` retrieves; `"ok"` or `"silent"` skips the network call entirely and forwards that book’s existing slot unchanged (see `CLAUDE.md`, "State invariants") — the skip a later retry round needs, so an already-good book’s retrieval isn’t redone.

In [1]:
# [C.2a]
# real call, no server needed: label='ok' takes the skip branch before any network call happens
import asyncio
from ParallelProse.agent import retriever, bridged_tools_by_corpus, retrieval_agents_by_corpus

bridged_tools_by_corpus['A'] = object()       # stand in for the real cached tools
retrieval_agents_by_corpus['A'] = object()    # stand in for the real cached inner agent

state = {
    'query': 'does not matter once a book is ok',
    'answer': '', 'needs_revision': False, 'feedback': '', 'iteration': 2,
    'corpora': {
        'A': {'chunks': ['already have these'],
              'lineage': [{'iteration': 0, 'tool': 'x', 'args': {}, 'forced_retriever': False}],
              'label': 'ok', 'reason': 'looked fine', 'narrowed_query': None},
    }
}
print(asyncio.run(retriever(state)))

{'corpora': {'A': {'chunks': ['already have these'], 'lineage': [{'iteration': 0, 'tool': 'x', 'args': {}, 'forced_retriever': False}], 'label': 'ok', 'reason': 'looked fine', 'narrowed_query': None}}}


<!-- [D-a] -->

## D — reflect as a diagnostic router, built

A binary pass/fail can’t tell "retrieval missed it" apart from "the book doesn’t address it" — and those two need opposite fixes. `reflect` now judges each book separately and names which one applies.

<!-- [D.1a] -->

`CorpusCritique` (agent.py:49-60) is one book’s verdict: `label` (`ok`/`miss`/`silent`), `reason` — the chunk-level evidence for that label — and `narrowed_query`, code-enforced back to `None` whenever `label != "miss"` (agent.py:303-304, 311-312), never left to the field description alone. `Critique` (agent.py:63-70) nests one `CorpusCritique` per book under a single `needs_revision`/`feedback` pair — one LLM call judges both books against the real retrieved chunks (`format_context`, agent.py:182-194) and the queries already tried (`format_attempts`, agent.py:226-241), not from the draft’s prose alone.

In [1]:
# [D.2a]
from ParallelProse.agent import CorpusCritique, Critique

print(Critique(
    needs_revision=True,
    feedback='Book B needs another look.',
    corpora_critique=[
        CorpusCritique(corpus_id='A', label='ok', reason='chunks match the claim', narrowed_query=None),
        CorpusCritique(corpus_id='B', label='miss', reason='right topic, wrong passage',
                        narrowed_query='a narrower version of the question'),
    ],
))

needs_revision=True feedback='Book B needs another look.' corpora_critique=[CorpusCritique(corpus_id='A', label='ok', reason='chunks match the claim', narrowed_query=None), CorpusCritique(corpus_id='B', label='miss', reason='right topic, wrong passage', narrowed_query='a narrower version of the question')]


<!-- [E-a] -->

## E — retry aimed at only the flagged book, built

Section C’s skip only matters if something upstream actually decides, per book, whether a retry round happens at all.

<!-- [E.1a] -->

`route_after_reflection` (agent.py:318-331) scans every book for `label == "miss"` — no `return` inside that scan, so a book’s position in `state["corpora"]` never decides the route. Any book still `miss` routes back to `retriever`, which (section C) only actually re-retrieves for that book; once no book is `miss`, the route falls straight to `END`, regardless of `needs_revision`.

In [1]:
# [E.2a]
from ParallelProse.agent import route_after_reflection

def mk(iteration, labels):
    return {'iteration': iteration, 'corpora': {cid: {'label': l} for cid, l in labels.items()}}

print('one book miss  ->', route_after_reflection(mk(1, {'A': 'ok', 'B': 'miss'})))
print('no book miss   ->', route_after_reflection(mk(1, {'A': 'ok', 'B': 'ok'})))
print('max iterations ->', route_after_reflection(mk(5, {'A': 'ok', 'B': 'miss'})))

one book miss  -> retriever
no book miss   -> __end__
END - MAX_ITERATIONS reached
max iterations -> __end__


<!-- [F-a] -->

## F — a structured comparison from composer, built

A free-prose paragraph can’t be checked against reflect’s per-book labels. composer needed slots reflect could actually verify.

<!-- [F.1a] -->

`ComposerAnswer` (agent.py:82-89) splits the draft into a cross-book `agreement`/`disagreement` pair plus `unique_findings` (agent.py:73-79) — one `CorpusFinding` per book. When a book’s label is `"silent"`, `composer` overwrites that book’s `finding` to a fixed statement of absence in code (agent.py:239-244), never left to the model’s prose. `reflect` reads the whole thing back through `format_answer` (agent.py:244-249), which renders `agreement`/`disagreement` plus each book’s `finding` by name.

In [1]:
# [F.2a]
from ParallelProse.agent import ComposerAnswer, CorpusFinding, format_answer

answer = ComposerAnswer(
    agreement='Both discuss X.',
    disagreement='A says one thing, B says another.',
    unique_findings=[
        CorpusFinding(corpus_id='A', finding="A's own claim", quote='a supporting excerpt'),
        CorpusFinding(corpus_id='B', finding='query content is absent', quote=None),
    ],
)
print(format_answer(answer))

Agreement: Both discuss X.
Disagreement: A says one thing, B says another. 

Corpus_id: A = Finding: A's own claim
Corpus_id: B = Finding: query content is absent


<!-- [G-a] -->

## G — the whole loop, for real

Sections B through F are the five pieces; here they run together, end to end, on the two books in `catalog.py` today.

In [1]:
# [G.1a]
import asyncio
from ParallelProse.agent import run_query

result = asyncio.run(run_query('How is time experienced by people, and what shapes that experience?'))
answer = result['answer']
print('Agreement:', answer.agreement)
print('Disagreement:', answer.disagreement)
for f in answer.unique_findings:
    print(f'[{f.corpus_id}] {f.finding}')
    print(f'    quote: {f.quote}')

Agreement: Both texts explore the concept of time, emphasizing its subjective nature and the ways it impacts human experience. Augustine discusses time as a mental construct, shaped by memory, perception, and expectation, while Debord critiques commodified time, which is shaped by societal structures and economic factors.
Disagreement: Augustine presents time as an internal experience related to the soul and its understanding of past, present, and future, whereas Debord critiques the external, commodified nature of time in modern society, arguing it is shaped by economic production and consumer culture.
[A] Time is experienced as a mental construct, shaped by memory and expectation, rather than a physical entity.
    quote: "For these three do exist in some sort, in the soul, but otherwhere do I not see them; present of things past, memory; present of things present, sight; present of things future, expectation."
[B] Time is experienced as commodified and pseudocyclical, shaped by econ

<!-- [G.2a] -->

Both books landed `label == "ok"` on this pass — the retrieved chunks already supported each claim, so section E’s retry branch never fired. `quote` on each finding (visible above) is one of the two v3 additions mentioned in the intro — it rides on top of `CorpusFinding`’s v2 shape without changing it.